# Survival Analysis — Liver Cirrhosis (PBC trial)

Mayo Clinic PBC trial, 418 patients. Goal: understand how long patients survived
(`N_Days`, `Status`), then compare D-penicillamine vs. placebo.

Step 1 of this notebook: load the data and look at the two columns that matter
for survival analysis, before touching any statistics.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/cirrhosis.csv")
print(df.shape)
df.head()

## The two columns that matter here

- `N_Days` — days from enrollment to the recorded event
- `Status` — what that event was: `D` (death), `C` (censored — still alive when
  the study ended), `CL` (censored — alive but had a liver transplant)

Let's see how the 418 patients split across these three outcomes.

In [ ]:
df["Status"].value_counts()

## Defining the event for survival analysis

`lifelines` needs a numeric event indicator, not the `Status` string:
- `event = 1` → death (`D`) — the event we're studying
- `event = 0` → censored (`C` or `CL`) — patient's true survival time is unknown

Only 161/418 (38.5%) patients have `event = 1`; the rest are censored.

In [ ]:
df["event"] = (df["Status"] == "D").astype(int)
df["event"].value_counts()

## Kaplan-Meier curve — whole cohort

Now we do exactly what we worked out by hand on the 5-patient toy example,
but with `lifelines` on all 418 real patients:
- `durations` — how long each patient was observed (`N_Days`)
- `event_observed` — did we see the actual event (death), or was the patient censored (`event`)

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
kmf.fit(durations=df["N_Days"], event_observed=df["event"])

fig, ax = plt.subplots(figsize=(7, 5))
kmf.plot_survival_function(ax=ax)
ax.set_xlabel("Days since enrollment")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier — all patients")
plt.show()

print("Median survival time (days):", kmf.median_survival_time_)

**Result:** median survival time is **3395 days (≈9.3 years)** — half of the 418
patients were still alive (or event-free) beyond that point. The confidence band
widens past ~3000 days because fewer patients remain under observation that far
out, so the estimate there is less certain.

## Drug vs. Placebo — restrict to randomized patients

`Drug` is `NaN` for 106/418 patients who were observed but not randomized into
the trial. Comparing treatment vs. placebo only makes sense on the 312 patients
who actually were randomized — otherwise we'd be comparing apples to oranges.

In [ ]:
randomized = df[df["Drug"].notna()]
randomized["Drug"].value_counts()

## Two Kaplan-Meier curves — Drug vs. Placebo

Same method as before, fit separately on each group, then plot both on one
chart so we can compare them visually.

In [ ]:
drug_group = randomized[randomized["Drug"] == "D-penicillamine"]
placebo_group = randomized[randomized["Drug"] == "Placebo"]

kmf_drug = KaplanMeierFitter()
kmf_placebo = KaplanMeierFitter()

fig, ax = plt.subplots(figsize=(7, 5))

kmf_drug.fit(drug_group["N_Days"], drug_group["event"], label="D-penicillamine")
kmf_drug.plot_survival_function(ax=ax)

kmf_placebo.fit(placebo_group["N_Days"], placebo_group["event"], label="Placebo")
kmf_placebo.plot_survival_function(ax=ax)

ax.set_xlabel("Days since enrollment")
ax.set_ylabel("Survival probability")
ax.set_title("Kaplan-Meier — Drug vs. Placebo")
plt.show()

## Is the difference statistically significant? — log-rank test

Two curves can look slightly different just by chance. The log-rank test checks
whether the difference between the two survival curves is bigger than we'd
expect from random noise alone.

- `p < 0.05` → the difference is statistically significant (unlikely due to chance)
- `p >= 0.05` → we can't rule out that the difference is just random variation

In [ ]:
from lifelines.statistics import logrank_test

result = logrank_test(
    drug_group["N_Days"], placebo_group["N_Days"],
    event_observed_A=drug_group["event"], event_observed_B=placebo_group["event"],
)
print("p-value:", result.p_value)

**Result:** p-value = 0.75, far above the 0.05 threshold. There is **no
statistically significant difference** in survival between D-penicillamine
and placebo — the curves overlap heavily, and so do their confidence bands.
This matches the actual historical finding from the Mayo Clinic PBC trial:
D-penicillamine did not demonstrate a survival benefit over placebo.

## Cox Proportional Hazards — beyond just Drug

The log-rank test only compared Drug vs. Placebo in isolation, ignoring
everything else about the patient. A Cox model looks at several factors
**at the same time** — Age, Bilirubin, Albumin, Copper, Prothrombin time,
disease Stage, and Drug — and estimates how much each one moves the risk
of death, while holding the others constant. This also lets us check
whether the "no drug effect" finding still holds once we control for
these other factors.

In [ ]:
cols = ["N_Days", "event", "Age", "Bilirubin", "Albumin", "Copper", "Prothrombin", "Stage", "Drug"]
cox_df = randomized[cols].dropna().copy()
cox_df["Drug"] = (cox_df["Drug"] == "D-penicillamine").astype(int)
print(cox_df.shape)

Unlike Kaplan-Meier, `CoxPHFitter` needs a complete row for every covariate we
include — no `NaN`s. We dropped any patient missing one of these 7 columns, so
the row count above will be a bit below 312. `Drug` is now `1` = D-penicillamine,
`0` = Placebo, so the model can use it as a number.

Now fit the model and print the summary table:

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="N_Days", event_col="event")
cph.print_summary()

## How to read this table

- **`exp(coef)`** — the hazard ratio for that variable. `> 1` means higher
  values of that variable *increase* the risk of death; `< 1` means they
  *decrease* it (protective). `1.0` means no effect.
- **`p`** — same rule as before: `< 0.05` means the effect is statistically
  significant.

Look specifically at the `Drug` row: if its `p` is still well above 0.05 and
`exp(coef)` is close to 1, that confirms the log-rank result — no real drug
effect, even after controlling for age, disease severity, and lab values.